# Pricing a contract with four Monte Carlo estimators

Compare prices, fund deltas, estimator variances, standard errors and CPU times. Every model and numerical input is written below; no preset is loaded.

These examples use paper-calibrated financial/carbon inputs and an illustrative five-year contract. The small numerical budgets demonstrate the workflow; they are not a convergence study or the paper's full hedging experiment.

**Workspace and execution controls.** Keep this notebook in the `notebook/` folder. Start it from that folder or a subdirectory and select the `.venv-analysis` Python kernel described in [INSTALLATION.md](../INSTALLATION.md). The setup detects the project root and locates the compiled C++ runner. `RUN_SIMULATIONS=False` reloads a completed run with matching simulation inputs. Use a new `RUN_NAME` when changing inputs; `OVERWRITE_EXISTING=True` explicitly permits replacing that run's generated files. All paths and switches below are editable.

In [ ]:
from pathlib import Path
from copy import deepcopy
import json
import sys

SEARCH_START = Path.cwd().resolve()
PROJECT_ROOT = next(
    (parent for parent in (SEARCH_START, *SEARCH_START.parents)
     if (parent / 'include' / 'carbonfund').is_dir()
     and (parent / 'python' / 'carbonfund_experiments.py').is_file()),
    None,
)
if PROJECT_ROOT is None:
    raise FileNotFoundError('Start the notebook from the CarbonFund_UL folder or a subdirectory.')
BINARY = PROJECT_ROOT / 'build-paper-release' / ('carbonfund_experiment.exe' if sys.platform == 'win32' else 'carbonfund_experiment')
OUTPUT_ROOT = PROJECT_ROOT / 'results' / 'pricing_example'
RUN_NAME = 'cir_T5_01'
RUN_SIMULATIONS = True
OVERWRITE_EXISTING = False
SAVE_FIGURES = True
SHOW_FIGURES = True
PRINT_PROGRESS = True

sys.path.insert(0, str(PROJECT_ROOT / 'python'))
import numpy as np
import matplotlib.pyplot as plt
from IPython.display import display
import carbonfund_experiments as cfe

RUN_DIRECTORY = OUTPUT_ROOT / RUN_NAME
if not RUN_NAME or Path(RUN_NAME).name != RUN_NAME or RUN_NAME in ('.', '..'):
    raise ValueError('RUN_NAME must be one directory name.')
if RUN_SIMULATIONS and not BINARY.is_file():
    raise FileNotFoundError(f'Build carbonfund_experiment first: {BINARY}')
print('Python:', sys.executable)
print('Python interface:', Path(cfe.__file__).resolve())
print('C++ backend:', BINARY)
print('Output directory:', RUN_DIRECTORY)

**Market inputs.** The dimension is fixed at four risky assets. `risk_free_rate` and the four `expected_returns` are continuously compounded annual rates; `volatilities` are annual diffusion coefficients. The symmetric positive-definite `correlations` matrix has unit diagonal. The backend constructs the covariance from volatilities and correlations. Asset expected returns enter the fund allocation; valuation uses the risk-free financial drift.

In [ ]:
market = {
    'risk_free_rate': 0.05,
    'expected_returns': [0.25, 0.15, 0.10, 0.08],
    'volatilities': [0.30, 0.25, 0.20, 0.16],
    'correlations': [
        [1.0000, 0.4397, 0.3900, 0.3168],
        [0.4397, 1.0000, 0.2954, 0.3261],
        [0.3900, 0.2954, 1.0000, 0.3134],
        [0.3168, 0.3261, 0.3134, 1.0000],
    ],
}

**Carbon inputs and model choices.** `model` selects `CIR`, `OU` or `GeoOU`. All vectors have four entries, one per factor. CIR solves $dC_i=\kappa_i(\bar C_i-C_i)dt+\sigma_i\sqrt{C_i}\,dB_i$; `initial` is the initial intensity, `long_run_mean` is $\bar C$, `mean_reversion` is $\kappa$, and `volatility` is $\sigma$. This scheme requires nonnegative states and $\kappa_i\bar C_i\geq\sigma_i^2/4$.

OU uses the same input names with additive Gaussian noise and allows negative intensities; allocation uses their positive part. GeoOU applies OU dynamics to log intensity: change `model` to `GeoOU`, replace `long_run_mean` with `long_run_log_mean`, and supply the desired log means. GeoOU `initial` remains in positive level units. Volatility units differ between models, so model changes require corresponding parameter choices. Carbon factors and financial noise are independent.

In [ ]:
carbon = {
    'model': 'CIR',
    'initial': [5000.0, 4000.0, 3000.0, 1000.0],
    'long_run_mean': [2500.0, 2000.0, 1500.0, 500.0],
    'mean_reversion': [0.05, 0.05, 0.05, 0.05],
    'volatility': [3.0, 3.0, 3.0, 3.0],
}

**Fund preferences.** `risk_aversion` is the positive quadratic-risk parameter $\delta$, and `carbon_penalty` is the nonnegative carbon-intensity penalty $\alpha$. They determine the fund's investment allocation together with market inputs and current carbon intensities. The remainder of wealth is invested in the risk-free account.

In [ ]:
fund = {
    'risk_aversion': 1.0,
    'carbon_penalty': 0.0025,
}

**Contract inputs.** `fund_value` is the initial fund value $X_0$ and `maturity` is in years. The lower and upper guarantees start at `minimum_guarantee` and `maximum_guarantee` and grow at the two specified continuous rates. The benefit at payment time is the fund value bounded by those guarantees. The bounds below coincide at time zero and separate for positive times.

PE pays on survival to maturity; TI pays on death before maturity; EI combines the PE survival benefit and the TI death benefit multiplied by `endowment_death_benefit_fraction`. Benefits are discounted once. Guarantees remain fixed when differentiating with respect to the current fund value.

In [ ]:
contract = {
    'fund_value': 1.0,
    'maturity': 10.0,
    'minimum_guarantee': 1.0,
    'maximum_guarantee': 1.0,
    'minimum_guarantee_growth_rate': 0.05,
    'maximum_guarantee_growth_rate': 0.50,
    'endowment_death_benefit_fraction': 1.0,
}

**Mortality inputs.** For initial age $a$, the Gompertz-Makeham hazard is

$$\mu(a+t)=\xi+\frac{1}{b}\exp\!\left(\frac{a+t-m}{b}\right).$$

`xi` is the nonnegative Makeham constant, `m` the Gompertz location and `b > 0` its scale in years. The corrected paper calibration uses `xi = 0.0041959`. Mortality is independent of financial and carbon risk.

In [ ]:
mortality = {
    'xi': 0.0041959,
    'm': 79.6921211,
    'b': 11.5818911,
}

**Population inputs.** Choose one `CONTRACT_TYPE`: `PE`, `TI` or `EI`. This pricing example contains one policy with the selected age and notional. The returned price and delta include that notional. More individual policies of the same type can be listed explicitly; all share the contract terms.

In [ ]:
CONTRACT_TYPE = 'EI'
POLICYHOLDER_AGE = 60.0
POLICY_NOTIONAL = 1.0
population = [
    {'type': CONTRACT_TYPE, 'age': POLICYHOLDER_AGE, 'notional': POLICY_NOTIONAL},
]

**Pricing inputs.** `paths` is the trajectory budget for each configuration and repetition; `intervals` is the pricing time grid. `standard` simulates financial randomness, while `conditioned` integrates it analytically conditional on carbon. Both integrate mortality analytically. Suffix `_antithetic` advances paired trajectories using opposite Gaussian inputs, with one pair average per independent statistical observation.

`repetitions` creates independently seeded complete evaluations. The summary uses the first repetition's estimates and median CPU time; repetitions are not pooled. The two seeds must be distinct. With any antithetic configuration selected, odd path requests round up for all methods to maintain equal actual budgets. Standard TI/EI uses sequential fund paths; standalone PE needs only a terminal financial draw.

In [ ]:
pricing = {
    'contract': CONTRACT_TYPE,
    'paths': 10000,
    'intervals': 25,
    'configurations': [
        'standard',
        'standard_antithetic',
        'conditioned',
        'conditioned_antithetic',
    ],
    'repetitions': 3,
    'carbon_seed': 20261015,
    'financial_seed': 20261016,
}

**Analysis inputs.** `normalization` selects aggregate, per-policy or per-notional units; raw simulation files retain aggregate amounts. Means and standard errors divide by exposure; variances divide by exposure squared. `confidence_level` selects approximate Student t mean intervals. `comparison_family_size=0` uses the current run's contrast count; a larger declared family gives Bonferroni simultaneous intervals.

Quantiles and upper-tail expected-shortfall levels are probabilities strictly between zero and one. `bins`, `plot_range` and `zoom` control histograms and the optional inset. Figure formats, raster DPI and title are explicit. These settings can be changed when reloading saved results without rerunning C++.

In [ ]:
analysis = {
    'normalization': 'per_notional',
    'confidence_level': 0.95,
    'comparison_family_size': 0,
    'quantiles': [0.025, 0.50, 0.90, 0.95, 0.975, 0.99],
    'expected_shortfall_levels': [0.95, 0.99],
    'bins': 70,
    'zoom': None,
    'plot_range': None,
    'figure_formats': ['png', 'pdf'],
    'dpi': 180,
    'title': '',
}

**Configuration.** Assemble one complete schema-version-1 dictionary from the explicit input cells. Deep copies keep the run inputs separate from later edits. Validation catches inconsistent dimensions, model inputs, estimator names, stream seeds and analysis settings before launching the backend. The exact configuration is saved with the run.

In [ ]:
NOTES = 'Explicit pricing notebook example; illustrative numerical budgets.'
config = {
    'schema_version': 1,
    'experiment': 'pricing',
    'market': deepcopy(market),
    'carbon': deepcopy(carbon),
    'fund': deepcopy(fund),
    'contract': deepcopy(contract),
    'mortality': deepcopy(mortality),
    'population': deepcopy(population),
    'pricing': deepcopy(pricing),
    'analysis': deepcopy(analysis),
    'notes': NOTES,
}
cfe.validate_config(config)
print(json.dumps(config, indent=2))

**Run or reload.** Simulation is performed by the selected C++ executable. The interface writes the configuration, metadata, raw numerical output, statistical summaries and execution log. Existing nonempty output directories require explicit overwrite permission.

Reload mode checks that simulation inputs match the saved run; only analysis settings and descriptive notes may differ. Figures are generated in the final cell so their saving and display are controlled explicitly.

In [ ]:
if RUN_SIMULATIONS:
    result = cfe.run_experiment(
        config, RUN_DIRECTORY, binary=BINARY,
        overwrite=OVERWRITE_EXISTING, plots=False, progress=PRINT_PROGRESS,
    )
else:
    result = cfe.load_result(RUN_DIRECTORY)
    def simulation_inputs(cfg):
        return {key: value for key, value in cfg.items()
                if key not in ('analysis', 'notes')}
    if simulation_inputs(result.config) != simulation_inputs(config):
        raise ValueError('Current simulation inputs differ from the saved run.')
    result.config['analysis'] = deepcopy(analysis)
    result.save_analysis(plots=False)
print('Saved results:', result.directory)

**Pricing results.** The table contains price and delta estimates, observation variance, standard deviation, variance of the estimated mean, standard error, intervals and variance reduction relative to the standard estimator. For antithetic sampling, observation variance is the variance of pair averages and the sample count is the number of pairs. Equal-budget comparisons therefore use the variance of the estimated mean.

The second table compares paired estimates across independent repetitions. The third compares observed repetition dispersion with reported uncertainty. Three repetitions give only a small calibration sample. These intervals describe Monte Carlo uncertainty for the discretized model, rather than time-grid bias.

In [ ]:
display(result.summary())
display(result.comparisons())
display(result.repetition_statistics())
print('Pricing data:', result.directory / 'pricing.json')
print('Summary CSV:', result.directory / 'summary.csv')
print('Repetitions CSV:', result.directory / 'repetition_statistics.csv')

**Figures and exported files.** Create the configured plots from saved numerical results. `SAVE_FIGURES` writes the selected formats and `SHOW_FIGURES` displays them. The original CSV/JSON outputs remain available for MATLAB, R or later Python sessions. Reloading with `load_result` never invokes the C++ simulation. Plot ranges affect presentation; the summary statistics are computed from the complete retained sample.

In [ ]:
if SAVE_FIGURES or SHOW_FIGURES:
    figure = result.plot()
    if SAVE_FIGURES:
        for extension in analysis['figure_formats']:
            figure.savefig(
                result.directory / ('pricing.' + extension),
                dpi=analysis['dpi'],
            )
    if SHOW_FIGURES:
        plt.show()
    plt.close(figure)
for path in sorted(result.directory.iterdir()):
    if path.is_file():
        print(path.name)